# Clase 05 — Transformación de Datos (Transform)

**Asignatura:** Ingeniería de Datos  
**Docente:** Ing. Sergio Orozco

La **transformación** es la etapa más compleja y creativa del pipeline ETL.  
Su objetivo es convertir datos crudos en datos **limpios, consistentes, enriquecidos y listos para el análisis**.

| Sección | Tema |
|---------|------|
| **1** | Instalación e importación de librerías |
| **2** | Carga y diagnóstico del dataset crudo |
| **3** | Limpieza de datos |
| **4** | Normalización y estandarización |
| **5** | Enriquecimiento mediante JOIN |
| **6** | Derivación de columnas calculadas |
| **7** | Agregaciones y resumen |
| **8** | Pivot Table |
| **9** | Guardado de resultados |

**Tecnologías:** `pandas`, `numpy`

## 1. Instalación e Importación de Librerías

In [1]:
%pip install pandas numpy openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# ── Rutas de trabajo ──────────────────────────────────────────────────────────
DIR_INPUT  = Path("datos/input")
DIR_OUTPUT = Path("datos/output")
DIR_OUTPUT.mkdir(parents=True, exist_ok=True)

print("Librerías importadas correctamente.")
print(f"pandas  versión: {pd.__version__}")
print(f"numpy   versión: {np.__version__}")

Librerías importadas correctamente.
pandas  versión: 3.0.2
numpy   versión: 2.4.4


## 2. Carga y Diagnóstico del Dataset Crudo

Antes de transformar, debemos **entender qué tenemos**:  
tipos de datos, nulos, duplicados y valores inconsistentes.

> Los archivos de entrada se encuentran en `datos/input/`  
> y simulan datos tal como llegarían desde un sistema OLTP real.

In [10]:
# Cargar el dataset de ventas crudas
df_crudo = pd.read_csv(DIR_INPUT / "ventas_crudas.csv")

print("=== PRIMERAS FILAS ===")
df_crudo

=== PRIMERAS FILAS ===


,id_venta,fecha_venta,cliente,monto,moneda,id_producto,cantidad,descuento_pct
0,1.0,05/01/2026,María García,1500.0,ARS,101,2,0.10
1,2.0,07/01/2026,JUAN LOPEZ,2300.5,ars,102,1,0.00
2,3.0,10/01/2026,Ana Torres,850.0,ARS,103,1,0.05
3,4.0,12/01/2026,Diego Herrera,450.0,PESOS,104,2,0.00
4,5.0,15/01/2026,Luis Paz,1200.0,USD,105,2,0.15
5,6.0,18/01/2026,Carlos Ruiz,950.0,ARS,106,1,0.00
6,7.0,20/01/2026,SOFIA ROMERO,3200.0,usd,108,4,0.20
7,8.0,21/01/2026,Martina López,450.0,ARS,109,1,0.00
8,9.0,2026-02-03,Roberto Silva,600.0,ARS,107,2,0.05
9,10.0,2026-02-05,Carmen Díaz,1500.0,ARS,101,1,0.00


In [12]:
# ── Diagnóstico inicial ────────────────────────────────────────────────────────
print("=== TIPOS DE DATO ===")
print(df_crudo.dtypes)

print(f"\n=== DIMENSIONES ===")
print(f"Filas: {len(df_crudo)}  |  Columnas: {len(df_crudo.columns)}")

print(f"\n=== VALORES NULOS POR COLUMNA ===")
print(df_crudo.isnull().sum())

print(f"\n=== FILAS COMPLETAMENTE DUPLICADAS ===")
print(f"Duplicados: {df_crudo.duplicated().sum()}")

print(f"\n=== VALORES ÚNICOS EN 'moneda' ===")
print(df_crudo["moneda"].value_counts())

=== TIPOS DE DATO ===
id_venta         float64
fecha_venta          str
cliente              str
monto            float64
moneda               str
id_producto        int64
cantidad           int64
descuento_pct    float64
dtype: object

=== DIMENSIONES ===
Filas: 20  |  Columnas: 8

=== VALORES NULOS POR COLUMNA ===
id_venta         1
fecha_venta      0
cliente          1
monto            0
moneda           0
id_producto      0
cantidad         0
descuento_pct    0
dtype: int64

=== FILAS COMPLETAMENTE DUPLICADAS ===
Duplicados: 1

=== VALORES ÚNICOS EN 'moneda' ===
moneda
ARS      13
ars       2
USD       2
PESOS     1
usd       1
pesos     1
Name: count, dtype: int64


### Problemas detectados en el dataset

| # | Problema | Columna | Acción |
|---|----------|---------|--------|
| 1 | Filas duplicadas | Todas | `drop_duplicates()` |
| 2 | `id_venta` nulo | `id_venta` | `dropna(subset=...)` |
| 3 | Fechas con formatos distintos y fecha inválida | `fecha_venta` | `pd.to_datetime(..., errors='coerce')` |
| 4 | Nombres con espacios y mayúsculas inconsistentes | `cliente` | `.str.strip().str.title()` |
| 5 | `moneda` con variantes: `ars`, `PESOS`, `usd` | `moneda` | `.str.upper()` + `replace()` |
| 6 | Montos negativos | `monto` | Filtrar con condición |
| 7 | `cliente` nulo | `cliente` | Imputar con `"Desconocido"` |

## 3. Limpieza de Datos

Corregimos uno a uno todos los problemas detectados.  
Siempre trabajamos sobre una **copia** del dataset original para no perder los datos crudos.

In [13]:
# Trabajamos sobre una copia para preservar los datos originales
df = df_crudo.copy()
n_original = len(df)

# ── Paso 1: Eliminar filas completamente duplicadas ───────────────────────────
df = df.drop_duplicates()
print(f"[1] Duplicados eliminados   : {n_original - len(df)} fila(s)")

# ── Paso 2: Eliminar filas sin ID de venta (no identificables) ────────────────
antes = len(df)
df = df.dropna(subset=["id_venta"])
print(f"[2] Filas sin id_venta      : {antes - len(df)} fila(s) eliminadas")

# ── Paso 3: Convertir id_venta a entero ───────────────────────────────────────
df["id_venta"] = df["id_venta"].astype(int)

# ── Paso 4: Imputar cliente nulo con 'Desconocido' ────────────────────────────
nulos_cliente = df["cliente"].isna().sum()
df["cliente"] = df["cliente"].fillna("Desconocido")
print(f"[4] Clientes nulos imputados: {nulos_cliente} fila(s)")

# ── Paso 5: Normalizar nombre de cliente (strip + Title Case) ─────────────────
df["cliente"] = df["cliente"].str.strip().str.title()

# ── Paso 6: Normalizar moneda (mayúsculas + mapeo de variantes) ───────────────
mapa_monedas = {"PESOS": "ARS", "pesos": "ARS"}
df["moneda"] = df["moneda"].str.strip().str.upper().replace(mapa_monedas)
print(f"[6] Monedas normalizadas    : {df['moneda'].unique()}")

# ── Paso 7: Parsear fechas con manejo de errores ──────────────────────────────
# errors='coerce' convierte los valores inválidos en NaT (Not a Time)
df["fecha_venta"] = pd.to_datetime(df["fecha_venta"], dayfirst=True, errors="coerce")
fechas_invalidas = df["fecha_venta"].isna().sum()
df = df.dropna(subset=["fecha_venta"])
print(f"[7] Fechas inválidas elim.  : {fechas_invalidas} fila(s)")

# ── Paso 8: Eliminar montos negativos ─────────────────────────────────────────
antes = len(df)
df = df[df["monto"] >= 0]
print(f"[8] Montos negativos elim.  : {antes - len(df)} fila(s)")

print(f"\n=== RESULTADO ===")
print(f"Filas originales: {n_original}  →  Filas limpias: {len(df)}")
df

[1] Duplicados eliminados   : 1 fila(s)
[2] Filas sin id_venta      : 1 fila(s) eliminadas
[4] Clientes nulos imputados: 1 fila(s)
[6] Monedas normalizadas    : <ArrowStringArray>
['ARS', 'USD']
Length: 2, dtype: str
[7] Fechas inválidas elim.  : 6 fila(s)
[8] Montos negativos elim.  : 1 fila(s)

=== RESULTADO ===
Filas originales: 20  →  Filas limpias: 11


,id_venta,fecha_venta,cliente,monto,moneda,id_producto,cantidad,descuento_pct
0,1,2026-01-05,María García,1500.0,ARS,101,2,0.10
1,2,2026-01-07,Juan Lopez,2300.5,ARS,102,1,0.00
2,3,2026-01-10,Ana Torres,850.0,ARS,103,1,0.05
3,4,2026-01-12,Diego Herrera,450.0,ARS,104,2,0.00
4,5,2026-01-15,Luis Paz,1200.0,USD,105,2,0.15
5,6,2026-01-18,Carlos Ruiz,950.0,ARS,106,1,0.00
6,7,2026-01-20,Sofia Romero,3200.0,USD,108,4,0.20
7,8,2026-01-21,Martina López,450.0,ARS,109,1,0.00
11,12,2026-02-14,Laura Gómez,850.0,ARS,103,3,0.00
12,13,2026-02-20,Miguel Torres,450.0,ARS,104,1,0.00


## 4. Normalización y Estandarización

Normalizar significa **dar el mismo formato a valores equivalentes**.  
Es crítico cuando los datos provienen de múltiples fuentes con distintas convenciones.

En este caso, ya normalizamos `moneda` y `cliente` en el paso anterior.  
Ahora verificamos el resultado y mostramos el estado limpio del dataset.

In [14]:
# Verificación del estado de cada columna tras la limpieza
print("=== TIPOS DE DATO DESPUÉS DE LA LIMPIEZA ===")
print(df.dtypes)

print("\n=== VALORES ÚNICOS EN 'moneda' (normalizado) ===")
print(df["moneda"].value_counts())

print("\n=== RANGO DE FECHAS ===")
print(f"Desde : {df['fecha_venta'].min().date()}")
print(f"Hasta : {df['fecha_venta'].max().date()}")

print("\n=== ESTADÍSTICAS DE MONTO ===")
print(df["monto"].describe().round(2))

=== TIPOS DE DATO DESPUÉS DE LA LIMPIEZA ===
id_venta                  int64
fecha_venta      datetime64[us]
cliente                     str
monto                   float64
moneda                      str
id_producto               int64
cantidad                  int64
descuento_pct           float64
dtype: object

=== VALORES ÚNICOS EN 'moneda' (normalizado) ===
moneda
ARS    9
USD    2
Name: count, dtype: int64

=== RANGO DE FECHAS ===
Desde : 2026-01-05
Hasta : 2026-03-20

=== ESTADÍSTICAS DE MONTO ===
count      11.00
mean     1209.14
std       854.70
min       450.00
25%       650.00
50%       950.00
75%      1350.00
max      3200.00
Name: monto, dtype: float64


## 5. Enriquecimiento mediante JOIN

El dataset de ventas tiene el `id_producto`, pero no el nombre ni la categoría.  
Cargamos el catálogo de productos y hacemos un **LEFT JOIN** para enriquecer las ventas.

> **LEFT JOIN**: mantiene todas las filas de la tabla izquierda (ventas)  
> y agrega los datos de la derecha (productos) donde el `id_producto` coincida.

In [15]:
# Cargar el catálogo de productos
df_productos = pd.read_csv(DIR_INPUT / "catalogo_productos.csv")

print("=== CATÁLOGO DE PRODUCTOS ===")
df_productos

=== CATÁLOGO DE PRODUCTOS ===


,id_producto,nombre,categoria,proveedor,precio_lista
0,101,Laptop Básica,Computación,TechCo,1500.0
1,102,Monitor 24 pulgadas,Periféricos,ViewMax,2300.0
2,103,Teclado Mecánico,Periféricos,KeyMaster,850.0
3,104,Mouse Inalámbrico,Periféricos,TechCo,450.0
4,105,Auriculares USB,Audio,SoundPro,1200.0
5,106,Webcam HD,Periféricos,ViewMax,950.0
6,107,Disco SSD 1TB,Almacenamiento,DataStore,600.0
7,108,Placa de Video,Computación,GfxPro,3200.0
8,109,Hub USB 7 puertos,Periféricos,TechCo,450.0


In [16]:
# LEFT JOIN: enriquecer ventas con datos del catálogo
df = df.merge(df_productos, on="id_producto", how="left")

# Verificar que no quedaron productos sin match
sin_match = df["nombre"].isna().sum()
print(f"Ventas sin producto en catálogo: {sin_match}")

print("\n=== VENTAS ENRIQUECIDAS ===")
df[["id_venta", "cliente", "fecha_venta", "id_producto", "nombre", "categoria", "monto"]]

Ventas sin producto en catálogo: 0

=== VENTAS ENRIQUECIDAS ===


,id_venta,cliente,fecha_venta,id_producto,nombre,categoria,monto
0,1,María García,2026-01-05,101,Laptop Básica,Computación,1500.0
1,2,Juan Lopez,2026-01-07,102,Monitor 24 pulgadas,Periféricos,2300.5
2,3,Ana Torres,2026-01-10,103,Teclado Mecánico,Periféricos,850.0
3,4,Diego Herrera,2026-01-12,104,Mouse Inalámbrico,Periféricos,450.0
4,5,Luis Paz,2026-01-15,105,Auriculares USB,Audio,1200.0
5,6,Carlos Ruiz,2026-01-18,106,Webcam HD,Periféricos,950.0
6,7,Sofia Romero,2026-01-20,108,Placa de Video,Computación,3200.0
7,8,Martina López,2026-01-21,109,Hub USB 7 puertos,Periféricos,450.0
8,12,Laura Gómez,2026-02-14,103,Teclado Mecánico,Periféricos,850.0
9,13,Miguel Torres,2026-02-20,104,Mouse Inalámbrico,Periféricos,450.0


## 6. Derivación de Columnas Calculadas

Creamos columnas nuevas a partir de las existentes:  
totales, fechas derivadas y segmentación por rango de monto.

> **Regla:** nunca modifiques los valores originales. Crea columnas nuevas.

In [17]:
# ── Columnas financieras ───────────────────────────────────────────────────────
df["total_bruto"]     = df["monto"] * df["cantidad"]
df["descuento_monto"] = df["total_bruto"] * df["descuento_pct"]
df["total_neto"]      = df["total_bruto"] - df["descuento_monto"]

# ── Columnas temporales ────────────────────────────────────────────────────────
df["anio"]       = df["fecha_venta"].dt.year
df["mes"]        = df["fecha_venta"].dt.month
df["dia_semana"] = df["fecha_venta"].dt.day_name()
df["trimestre"]  = df["fecha_venta"].dt.quarter
df["anio_mes"]   = df["fecha_venta"].dt.to_period("M")   # Período año-mes

# ── Segmentación por monto neto ───────────────────────────────────────────────
# np.select permite asignar categorías según múltiples condiciones
condiciones = [
    df["total_neto"] < 1000,
    (df["total_neto"] >= 1000) & (df["total_neto"] < 4000),
    df["total_neto"] >= 4000,
]
categorias = ["Pequeña", "Mediana", "Grande"]
df["segmento_venta"] = np.select(condiciones, categorias, default="Sin clasificar")

print("=== COLUMNAS DERIVADAS ===")
df[["id_venta", "total_bruto", "descuento_monto", "total_neto",
    "dia_semana", "trimestre", "segmento_venta"]]

=== COLUMNAS DERIVADAS ===


,id_venta,total_bruto,descuento_monto,total_neto,dia_semana,trimestre,segmento_venta
0,1,3000.0,300.0,2700.0,Monday,1,Mediana
1,2,2300.5,0.0,2300.5,Wednesday,1,Mediana
2,3,850.0,42.5,807.5,Saturday,1,Pequeña
3,4,900.0,0.0,900.0,Monday,1,Pequeña
4,5,2400.0,360.0,2040.0,Thursday,1,Mediana
5,6,950.0,0.0,950.0,Sunday,1,Pequeña
6,7,12800.0,2560.0,10240.0,Tuesday,1,Grande
7,8,450.0,0.0,450.0,Wednesday,1,Pequeña
8,12,2550.0,0.0,2550.0,Saturday,1,Mediana
9,13,450.0,0.0,450.0,Friday,1,Pequeña


## 7. Agregaciones y Resumen

Las **agregaciones** consolidan múltiples registros en métricas resumidas.  
Son la base de los reportes y dashboards de negocio.

Calculamos ventas totales, ticket promedio y cantidad de transacciones  
agrupadas por **categoría de producto**.

In [18]:
# Agrupar por categoría y calcular métricas de negocio
resumen_categoria = (
    df.groupby("categoria")
    .agg(
        total_vendido    = ("total_neto",  "sum"),      # Suma de totales netos
        cant_transacc    = ("id_venta",    "count"),    # Cantidad de ventas
        ticket_promedio  = ("total_neto",  "mean"),     # Monto promedio por venta
        unidades_totales = ("cantidad",    "sum"),      # Unidades vendidas
    )
    .round(2)
    .sort_values("total_vendido", ascending=False)
    .reset_index()
)

print("=== VENTAS POR CATEGORÍA ===")
resumen_categoria

=== VENTAS POR CATEGORÍA ===


,categoria,total_vendido,cant_transacc,ticket_promedio,unidades_totales
0,Computación,13930.0,3,4643.33,7
1,Periféricos,8408.0,7,1201.14,10
2,Audio,2040.0,1,2040.00,2


In [19]:
# Agrupar por moneda para ver el total en cada divisa
resumen_moneda = (
    df.groupby("moneda")
    .agg(
        total_vendido = ("total_neto", "sum"),
        cant_ventas   = ("id_venta",   "count"),
    )
    .round(2)
    .reset_index()
)

print("=== VENTAS POR MONEDA ===")
resumen_moneda

=== VENTAS POR MONEDA ===


,moneda,total_vendido,cant_ventas
0,ARS,12098.0,9
1,USD,12280.0,2


## 8. Pivot Table

Una **tabla pivot** reorganiza los datos para comparar categorías y períodos  
en un formato matricial, muy usado en reportes de gerencia.

Construimos una tabla con **categorías en columnas** y **segmento de venta en filas**.

In [20]:
# Pivot: filas = segmento de venta, columnas = categoría de producto
tabla_pivot = df.pivot_table(
    index   = "segmento_venta",       # Filas
    columns = "categoria",            # Columnas
    values  = "total_neto",           # Valores a agregar
    aggfunc = "sum",                  # Función de agregación
    fill_value = 0,                   # Rellenar celdas vacías con 0
).round(2)

print("=== TOTAL NETO POR SEGMENTO DE VENTA Y CATEGORÍA ===")
tabla_pivot

=== TOTAL NETO POR SEGMENTO DE VENTA Y CATEGORÍA ===


categoria,Audio,Computación,Periféricos
segmento_venta,,,
Grande,0.0,10240.0,0.0
Mediana,2040.0,2700.0,4850.5
Pequeña,0.0,990.0,3557.5


## 9. Guardado de Resultados

Guardamos tres archivos en `datos/output/`:

| Archivo | Contenido |
|---------|----------|
| `ventas_transformadas.csv` | Dataset completo limpio y enriquecido |
| `resumen_por_categoria.csv` | Métricas agregadas por categoría |
| `pivot_segmento_categoria.csv` | Tabla pivot para reportes |

In [21]:
# ── 1. Dataset completo transformado ──────────────────────────────────────────
ruta_ventas = DIR_OUTPUT / "ventas_transformadas.csv"
df.to_csv(ruta_ventas, index=False, encoding="utf-8")
print(f"Guardado: {ruta_ventas}  ({len(df)} filas)")

# ── 2. Resumen por categoría ───────────────────────────────────────────────────
ruta_resumen = DIR_OUTPUT / "resumen_por_categoria.csv"
resumen_categoria.to_csv(ruta_resumen, index=False, encoding="utf-8")
print(f"Guardado: {ruta_resumen}  ({len(resumen_categoria)} filas)")

# ── 3. Tabla pivot ────────────────────────────────────────────────────────────
ruta_pivot = DIR_OUTPUT / "pivot_segmento_categoria.csv"
tabla_pivot.to_csv(ruta_pivot, encoding="utf-8")
print(f"Guardado: {ruta_pivot}")

print("\n✓ Todos los archivos guardados exitosamente en datos/output/")

Guardado: datos\output\ventas_transformadas.csv  (11 filas)
Guardado: datos\output\resumen_por_categoria.csv  (3 filas)
Guardado: datos\output\pivot_segmento_categoria.csv

✓ Todos los archivos guardados exitosamente en datos/output/


## Resumen de la Clase

| Técnica | Función pandas / numpy | Para qué sirve |
|---------|------------------------|----------------|
| Eliminar duplicados | `drop_duplicates()` | Evitar doble conteo |
| Eliminar nulos clave | `dropna(subset=[...])` | Filas no identificables |
| Imputar nulos | `fillna(valor)` | Preservar filas con datos parciales |
| Normalizar texto | `.str.strip().str.title()` | Consistencia de nombres |
| Parsear fechas | `pd.to_datetime(..., errors='coerce')` | Unificar formatos de fecha |
| Filtrar valores | `df[condición]` | Eliminar registros inválidos |
| Reemplazar valores | `.replace(diccionario)` | Mapear variantes a valor estándar |
| JOIN | `df.merge(otro, on=col, how='left')` | Enriquecer con tablas de referencia |
| Columnas calculadas | Operaciones aritméticas sobre columnas | Totales, descuentos, márgenes |
| Fechas derivadas | `.dt.year`, `.dt.month`, `.dt.quarter` | Análisis temporal |
| Segmentación | `np.select(condiciones, categorias)` | Clasificar registros por rangos |
| Agregaciones | `.groupby().agg()` | Métricas de negocio |
| Tabla pivot | `df.pivot_table()` | Reportes matriciales |

> **Regla de oro:** Siempre trabajar sobre una copia (`df.copy()`)  
> y guardar el dataset crudo sin modificaciones para poder reprocessarlo si es necesario.